# Regular Positive and Negative Inference (RPNI)

This notebook demonstrates the **RPNI algorithm**, a state-merging algorithm for learning finite-state automata from positive and negative examples.

## Algorithm Overview

RPNI (Regular Positive and Negative Inference) learns a deterministic finite automaton (DFA) by:

1. **Building a prefix tree** from positive examples (words to accept)
2. **Iteratively merging states** that are compatible with both positive and negative examples
3. **Producing a minimal DFA** that generalizes from the training data

The algorithm ensures that:
- All positive examples are accepted
- All negative examples are rejected
- The resulting automaton is as general (minimal) as possible

In [1]:
from itertools import count

from state_merging.algorithms.rpni import rpni
from state_merging.automata.DFA import DFA, assert_DFA
from state_merging.automata.SFST import run
from state_merging.collections.dense_int_dict import DenseIntDict
from state_merging.collections.dense_int_set import DenseIntSet

Create positive examples (strings that should be accepted) and negative examples (strings that should be rejected). The RPNI algorithm uses both to learn a DFA that generalizes correctly.

In [2]:
input_set: set[int] = {0, 1}

pos_dataset: list[list[int]] = [
    [1],
    [1, 0, 0],
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 1],
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1],
    [1, 0, 1, 0, 1]
]

neg_dataset: list[list[int]] = [
    [],
    [0],
    [0, 0],
    [1, 0],
    [0, 1],
    [1, 1],
    [0, 0, 0],
    [1, 1, 0],
    [1, 0, 1],
    [0, 1, 1]
]

Run the RPNI algorithm with our training data. Then validate that the learned DFA accepts all positive examples and rejects all negative examples.

In [3]:
dfa: DFA[int, int]
dfa, _ = rpni(
    input_set=input_set,
    pos_dataset=pos_dataset,
    neg_dataset=neg_dataset,
    choose_transition=lambda _, trs: next(iter(trs)),
    search_iter=lambda _, qs: qs,
    state_supply=count(),
    empty_fst_state_set=DenseIntSet(),
    empty_transition_mapping={},
    empty_final_output_mapping=DenseIntDict(),
    make_empty_visited_state_set=lambda qs: DenseIntSet(size=len(qs)),
    make_default_populated_mapping=lambda qs: DenseIntDict(size=len(qs), init=list),
    verbose=True
)

assert_DFA(dfa)

for d in pos_dataset:
    assert run(dfa, d, None, lambda none, _: none) is not None, \
        f"learned DFA rejected positive data {d}"

for d in neg_dataset:
    assert run(dfa, d, None, lambda none, _: none) is None, \
        f"learned DFA accepted negative data {d}"

with the following negative data: [
	[], None
	[0], None
	[0, 0], None
	[1, 0], None
	[0, 1], None
	[1, 1], None
	[0, 0, 0], None
	[1, 1, 0], None
	[1, 0, 1], None
	[0, 1, 1], None
]

learning from the following positive data by state-merging: [
	[1], None
	[1, 0, 0], None
	[0, 1, 0], None
	[0, 0, 1], None
	[1, 1, 1], None
	[1, 0, 0, 0, 0], None
	[0, 1, 0, 0, 0], None
	[0, 0, 1, 0, 0], None
	[0, 0, 0, 1, 0], None
	[0, 0, 0, 0, 1], None
	[1, 0, 1, 0, 1], None
]

naively constructed PTT:
SFST(state_set=DenseIntSet({0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24}), input_set={0, 1}, initial_state=0, transitions={(0, 1): (1, None), (1, 0): (2, None), (2, 0): (3, None), (0, 0): (4, None), (4, 1): (5, None), (5, 0): (6, None), (4, 0): (7, None), (7, 1): (8, None), (1, 1): (9, None), (9, 1): (10, None), (3, 0): (11, None), (11, 0): (12, None), (6, 0): (13, None), (13, 0): (14, None), (8, 0): (15, None), (15, 0): (16, None), (7, 0): (17, None), (17, 1)